# HW5 Mason Holcombe

In [79]:
import gensim
import gensim.downloader as api

from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import pandas as pd


# 1)

In [2]:
EMBEDDING_DIM = 300
WINDOW_SIZE = 3

dataset = api.load("text8")
model = gensim.models.Word2Vec(sentences=dataset,
                               vector_size=EMBEDDING_DIM,
                               window=WINDOW_SIZE)

print(f"STEP 1 | Completed {EMBEDDING_DIM} dim training.")

STEP 1 | Completed 300 dim training.


In [3]:
similar_words = [
    ("king", "queen"),
    ("fast", "quick"),
    ("begin", "start"),
    ("ocean", "sea"),
    ("hot", "warm")
]

print(f"STEP 1 | Similarity Scores:")
for w1, w2 in similar_words:
    wv1 = model.wv[w1].reshape(1, -1)
    wv2 = model.wv[w2].reshape(1, -1)

    sim_score = cosine_similarity(wv1, wv2).item()

    print(f"\t({w1}, {w2}) -> {sim_score:.4f}")

STEP 1 | Similarity Scores:
	(king, queen) -> 0.6330
	(fast, quick) -> 0.6597
	(begin, start) -> 0.6697
	(ocean, sea) -> 0.5789
	(hot, warm) -> 0.7549


# 2)

In [4]:
print(f"STEP 2 | Downloading google-news-300...")
pretrained_model = api.load('word2vec-google-news-300')
print(f"STEP 2 | Download complete.")

STEP 2 | Downloading google-news-300...
STEP 2 | Download complete.


In [ ]:
words = ["neural", "calculus", "happiness", "flower"]

print(f"STEP 2 | Large model qualitative behavior:")
for word in words:
    top5 = pretrained_model.most_similar(word, topn=5)
    
    print(f"\nTop 5 words similar to '{word}':")
    for i, (w, score) in enumerate(top5, start=1):
        print(f"{i}. {w:^18} {score:.4f}")


STEP 2 | Large model qualitative behavior:

Top 5 words similar to 'neural':
1.      neuronal      0.7805
2.      neurons       0.7326
3.  neural_circuits   0.7253
4.       neuron       0.7174
5.      cortical      0.6941

Top 5 words similar to 'calculus':
1.      algebra       0.6411
2.      Calculus      0.5957
3.        math        0.5777
4.    trigonometry    0.5716
5.    precalculus     0.5526

Top 5 words similar to 'happiness':
1.    contentment     0.7695
2.        joy         0.6183
3.     Happiness      0.6116
4.      hapiness      0.5749
5.   contentedness    0.5575

Top 5 words similar to 'flower':
1.       floral       0.7494
2.      flowers       0.7489
3.       roses        0.6977
4.       orchid       0.6929
5.       tulip        0.6629


# 3)

In [93]:
wordsim353 = pd.read_csv("wordsim_similarity_goldstandard.csv")
wordsim353.columns = ["word1", "word2", "mean_human_score"]

In [95]:
def pretrained_model_similarity(pretrained_model, word1, word2):
    wv1 = pretrained_model[word1].reshape(1, -1)
    wv2 = pretrained_model[word2].reshape(1, -1)

    sim_score = cosine_similarity(wv1, wv2).item()

    return sim_score

wordsim353["model_score"] = wordsim353.apply(lambda row: pretrained_model_similarity(pretrained_model, row["word1"], row["word2"]), axis=1)

In [96]:
rank_coef = spearmanr(wordsim353["mean_human_score"], wordsim353["model_score"]).statistic.item()
print(f"STEP 3 | Spearman's Rank Correlation Coefficient on WordSim-353: {rank_coef:.4f}")

STEP 3 | Spearman's Rank Correlation Coefficient on WordSim-353: 0.7000


# 4)